## <a name="get_to_know_coco_dataset">1. 认识COCO数据集</a>


本实践使用由微软主导发布的[COCO数据集](http://cocodataset.org)，该数据集是一个大型图像数据集, 专为对象检测、分割、人体关键点检测、语义分割和字幕生成而设计。得到了业界相关公司和机构的广泛支持。在COCO数据库中，图片被标注出bouding box以及mask，用于相关模型的训练。COCO本身还提供了[COCO API](https://github.com/cocodataset/cocoapi)，用于数据集的加载、标注信息解析以及可视化等。示例项目中对COCO API进行了进一步的封装，使得调试数据集变得更加方便。下面我们加载COCO数据集，并对数据集进行一些基本的操作。

### 准备数据集

在之前的步骤中，我们已经在data/目录下准备好最小COCO数据集，数据集目录结构如下：


```bash
data/
├── annotations
│   ├── instances_train2014.json
│   ├── ***.json
├── train2014
│   ├── COCO_train2014_***.jpg
└── val2014
    ├── COCO_val2014_***.jpg
```

接下来，我们进行代码实践

第一步：引入Mask R-CNN相关库和COCO的封装，注意指定的COCO_DIR与实际环境的地址一致

In [ ]:
import sys
import itertools
import math
import logging
import json
import re
import random
from collections import OrderedDict
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.lines as lines
from matplotlib.patches import Polygon

from src.mrcnn import utils
from src.mrcnn import visualize
from src.mrcnn.visualize import display_images
import src.mrcnn.model as modellib
from src.mrcnn.model import log

%matplotlib inline

from src.mrcnn import coco
config = coco.CocoConfig()
COCO_DIR = "data"  # 注意指定实际的数据集根目录地址

In [ ]:
dataset = coco.CocoDataset()
dataset.load_coco(COCO_DIR, "train") #　加载训练数据集
dataset.prepare()

In [ ]:
# 查看数据集中的图片数量和分类信息
print("Image Count: {}".format(len(dataset.image_ids)))
print("Class Count: {}".format(dataset.num_classes))
cols = 4
num_classes = len(dataset.class_info)

for i in range(0, num_classes, cols):
    line = ''
    for j in range(cols):
        index = i + j
        if index < num_classes:
            space = '\t' * (index % cols)
            info = dataset.class_info[i+j]
            line += "{}{:2}. {:12}".format(space, i+j, info['name'])
    print(line)

第二步：从训练数据中随机抽选1条，显示图片，Mask，以及Mask相应的分类id和分类名称

In [ ]:
image_id = np.random.choice(dataset.image_ids)
image = dataset.load_image(image_id)
mask, class_ids = dataset.load_mask(image_id)
visualize.display_top_masks(image, mask, class_ids, dataset.class_names, limit=2)

第三步：显示物体的包围框。这里我们不直接使用COCO数据库中标注的包围框，而是直接由Mask生成包围框，这样可以保证包围框和Mask的一致性，也可以更方便的对图片进行尺寸调整、裁剪等操作

In [ ]:
# 随机选出一张图片
# image_id = random.choice(dataset.image_ids)
image = dataset.load_image(image_id)
mask, class_ids = dataset.load_mask(image_id)
# 计算包围框
bbox = utils.extract_bboxes(mask)

# 显示图片及其信息
print("image_id ", image_id, dataset.image_reference(image_id))
log("image", image)
log("mask", mask)
log("class_ids", class_ids)
log("bbox", bbox)
# 显示图片中的物体实例
visualize.display_instances(image, bbox, mask, class_ids, dataset.class_names)

### 调整图片尺寸
因为我们要在同一个batch中处理多条数据，因此需要将图片调整为统一的尺寸。本示例中为(1024x1024)。图片的原始比例将得到保留，如果原图比例不是1:1，上下左右多出的空白区域将使用０进行填充（显示为黑色）

In [ ]:
# 随机选择一张图片
# image_id = np.random.choice(dataset.image_ids, 1)[0]
image = dataset.load_image(image_id)
mask, class_ids = dataset.load_mask(image_id)
original_shape = image.shape
# 对图片进行裁剪
image, window, scale, padding, _ = utils.resize_image(
    image, 
    min_dim=config.IMAGE_MIN_DIM, 
    max_dim=config.IMAGE_MAX_DIM,
    mode=config.IMAGE_RESIZE_MODE)
mask = utils.resize_mask(mask, scale, padding)
# 计算包围框
bbox = utils.extract_bboxes(mask)

# 显示图片，图片信息以及图中物体的实例
print("image_id: ", image_id, dataset.image_reference(image_id))
print("Original shape: ", original_shape)
log("image", image)
log("mask", mask)
log("class_ids", class_ids)
log("bbox", bbox)
visualize.display_instances(image, bbox, mask, class_ids, dataset.class_names)

### 最小化Mask
第一步：优化Mask

对于大尺寸的图片，如示例中的1024x1024的图片，一个Mask占用的内存会比较大，而如果图片中需要标注的物体越多，全图所需要保存的Mask就会使用越多的内存。对Mask我们进行如下优化：
- 仅保存包围框内的Mask
- 缩小Mask的尺寸，如56x56。缩小Mask会丢失一部分精度，但标注本身包含的Mask信息也并不是非常精确的，因此，缩小Mask带来的损失是可以接受的

In [ ]:
# image_id = np.random.choice(dataset.image_ids, 1)[0]
image, image_meta, class_ids, bbox, mask = modellib.load_image_gt(
    dataset, config, image_id, use_mini_mask=False)

log("image", image)
log("image_meta", image_meta)
log("class_ids", class_ids)
log("bbox", bbox)
log("mask", mask)

display_images([image]+[mask[:,:,i] for i in range(min(mask.shape[-1], 7))])

visualize.display_instances(image, bbox, mask, class_ids, dataset.class_names)

第二步：显示Mask并应用到原图，对比损失的精度。可以看到，损失在可以接受的范围之内

In [ ]:
# Add augmentation and mask resizing.
image, image_meta, class_ids, bbox, mask = modellib.load_image_gt(
    dataset, config, image_id, augment=True, use_mini_mask=True)
log("mask", mask)
display_images([image]+[mask[:,:,i] for i in range(min(mask.shape[-1], 7))])

mask = utils.expand_mask(bbox, mask, image.shape)
file_save_path = "01.result.jpg"
visualize.display_instances(image, bbox, mask, class_ids, dataset.class_names, save_path=file_save_path)